# Coleta de dados do vlr.gg — TCC (Google Colab)

Este notebook roda o scraper do projeto `vlr_scraper` no Colab e grava **tudo numa pasta do Google Drive**:

| Pasta no Drive | Conteúdo | Formato |
|---|---|---|
| `vlr_dados/raw/` | páginas baixadas (cache da coleta, permite retomar) | `.html.gz` |
| `vlr_dados/processed/` | tabelas finais (`rounds`, `mapas`, `partidas`...) | Parquet |
| `vlr_dados/amostra/` | tabelas de exemplo, para inspeção rápida | CSV |
| `vlr_dados/logs/` | log de cada execução e `falhas.parquet` | texto / Parquet |

**Antes de começar:** envie a pasta `vlr_scraper` (ou o `vlr_scraper.zip`) para `MyDrive/TCC/` no Google Drive.

**Se o Colab desconectar:** rode de novo as células 1 a 4 e depois a célula de coleta. O que já está em `raw/` não é baixado outra vez.

## 1. Montar o Google Drive e definir as pastas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA_TCC    = '/content/drive/MyDrive/TCC'
PASTA_CODIGO = f'{PASTA_TCC}/vlr_scraper'      # pasta do projeto no Drive (ou vlr_scraper.zip)
PASTA_DADOS  = f'{PASTA_TCC}/vlr_dados'        # onde ficam todas as saídas

## 2. Copiar o código para o Colab e instalar dependências

O código é copiado para `/content` (disco local do Colab, mais rápido). Os **dados** continuam indo para o Drive.

In [ ]:
import os, shutil, sys, zipfile

LOCAL = '/content/vlr_scraper'
shutil.rmtree(LOCAL, ignore_errors=True)
if os.path.isdir(PASTA_CODIGO):
    shutil.copytree(PASTA_CODIGO, LOCAL, ignore=shutil.ignore_patterns('data', '__pycache__', '.pytest_cache'))
elif os.path.exists(PASTA_CODIGO + '.zip'):
    with zipfile.ZipFile(PASTA_CODIGO + '.zip') as z:
        z.extractall('/content')
else:
    raise FileNotFoundError(f'Envie a pasta vlr_scraper ou vlr_scraper.zip para {PASTA_TCC}')

%cd /content/vlr_scraper
!pip install -q -r requirements.txt
if LOCAL not in sys.path:
    sys.path.insert(0, LOCAL)

## 3. Rodar os testes (opcional, ~10 s)

Confere o parser contra páginas reais salvas no projeto. Não acessa o site e não grava no Drive.

In [ ]:
!python -m pytest -q tests

## 4. Preparar a pasta de dados

`min_delay` e `max_delay` são o intervalo (em segundos) entre requisições ao vlr.gg. O padrão de 3 a 6 s é o mais seguro para coletas longas.

In [ ]:
from vlr import colab
colab.preparar(PASTA_DADOS, min_delay=3, max_delay=6)

## 5. Amostra: ver o que é extraído, sem acessar o site

Gera em `vlr_dados/amostra/` os mesmos arquivos da coleta real, **em CSV**, a partir de duas partidas reais salvas no projeto.

In [ ]:
amostra = colab.amostra()
amostra['rounds'].head()

In [ ]:
amostra['mapas']

## 6. Listar os eventos concluídos (todas as páginas do tier 60)

In [ ]:
eventos = colab.listar_eventos()
eventos[['event_id', 'event_name', 'dates', 'region', 'page']]

## 7. Sondagem de 2D replay (opcional, demorada)

Testa a 1ª e a última partida de cada evento. Para todos os eventos, leva ~1 a 2 h com o intervalo padrão. O resultado já está em `docs/sondagem_replay.csv` (out/2026); rode só se quiser atualizar.

In [ ]:
# sondagem = colab.sondar()                      # todos os eventos
# sondagem = colab.sondar(eventos=[2766, 2977])  # só alguns

## 8. Teste curto de coleta

Baixa só 2 partidas de um evento e gera os Parquet em `vlr_dados/processed/`. Confira o resultado antes da coleta completa.

In [ ]:
colab.coletar([2766], max_partidas=2)
colab.ler('mapas')

In [ ]:
colab.ler('rounds').head()

## 9. Coleta completa

Coloque em `EVENTOS` os IDs do recorte do TCC (veja a tabela da célula 6 ou `docs/sondagem_replay.csv`).

A célula pode rodar por horas. Se o Colab desconectar, rode de novo as células 1, 2 e 4 e depois esta: a coleta continua de onde parou.

In [ ]:
EVENTOS = [
    2766,   # Valorant Champions 2026
    # 2977, # VCT 2026: Americas Stage 2
    # 2976, # VCT 2026: EMEA Stage 2
    # 2776, # VCT 2026: Pacific Stage 2
]
resumo = colab.coletar(EVENTOS)
resumo

## 10. Conferir o resultado

In [ ]:
colab.ler('cobertura_por_evento')

In [ ]:
colab.falhas()        # partidas/eventos que falharam (rode a coleta de novo para refazê-los)

In [ ]:
colab.uso_de_espaco()  # tamanho de cada pasta no Drive

Para regerar as tabelas a partir do cache (por exemplo, depois de trocar a tabela de preços em `config/`), sem baixar nada:

In [ ]:
# colab.processar()

## 11. Encerrar

Garante que todos os arquivos foram enviados ao Drive antes de fechar o notebook.

In [ ]:
colab.finalizar()